# Поиск 50 кандидатов для запросов Авито

Для каждого запроса собираю кандидатов из нескольких источников: текстового поиска, похожих заголовков, истории запросов, близких микрокатегорий и географии. Затем `HistGradientBoostingClassifier` выбирает 50 объявлений из общего пула.

### Данные и признаки
Использую `train.parquet` (история взаимодействий), `benchmark_queries.parquet` (запросы для ответа) и `benchmark_items.parquet` (допустимые объявления). Для сравнения текстов объединяю заголовок, параметры и часть описания. Отдельно считаю символьное сходство заголовка, совпадения слов и фильтров. История помогает находить уже известные объявления, микрокатегория - переносить сигнал на новые, а расстояние и совпадение локации - учитывать город без жёсткого отсечения удалённых услуг. Добавляю цену, рейтинг и признаки доступности связи. `item_id` не используется как входной признак модели.

### Проверка качества
Из истории откладываю 2200 контекстов: для новых текстов удаляю все взаимодействия с ними, для известных удаляю выбранный контекст. Исторические индексы строятся уже после этого исключения. На первых 1400 запросах обучаю модель, следующие 800 использую для сравнения по среднему Recall@50. Локально: простая формула - **0.731875**, бустинг с базовыми признаками - **0.813083**, с дополнительными - **0.829583**; recall общего пула - **0.903125**.

### Что выяснилось при проверке
Часть релевантных объявлений терялась при поиске только по словам или без учёта города - добавила символьный и локальный каналы. История хуже помогает с новыми объявлениями, поэтому использую поиск по текстам и близкие микрокатегории. Отдельно исправила утечку из отложенной выборки в исторические индексы. Остались потери как при сборе кандидатов, так и при отборе финальных 50.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

import gc
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

DATA_DIR = Path('dataset')
OUTPUT_PATH = Path('answer.csv')

SEARCH = ['search_query', 'search_location_id', 'search_is_delivery_search',
          'search_infm_params_text', 'search_category']

## 1. Подготовка текста и дополнительные признаки

Привожу текст к нижнему регистру, заменяю `ё` на `е` и убираю пунктуацию. Большой файл истории читаю частями, чтобы не держать лишние столбцы в памяти.

In [2]:
def norm(x):
    return re.sub('[^а-яa-z0-9]+', ' ', str(x or '').lower().replace('ё', 'е')).strip()

def log(s):
    print(time.strftime('%H:%M:%S'), s, flush=True)

def top(x, k):
    k = min(k, len(x))
    ids = np.argpartition(x, -k)[-k:]
    return ids[np.argsort(-x[ids], kind='stable')]

def texts(path, ids=None):
    out = []
    for b in pq.ParquetFile(path).iter_batches(batch_size=4000):
        d = b.to_pandas()
        if ids is not None:
            d = d[d.item_id.isin(ids)]
        d = d.drop_duplicates('item_id')
        d['text'] = d.item_title_raw.fillna('') + ' ' + d.item_infm_params_text.fillna('').str[:600] + ' ' + d.item_description_raw.fillna('').str[:1600]
        d['item_infm_params_text'] = d.item_infm_params_text.fillna('').str[:1200]
        out.append(d.drop(columns=['item_description_raw']))
    return pd.concat(out, ignore_index=True).drop_duplicates('item_id')

Доля совпавших слов и наличие всей фразы помогают отличить частичное текстовое совпадение от более полного. Префиксы слов использую как простую поправку на разные окончания; это не полноценная лемматизация. Фильтры учитываются как признаки, а не как строгие условия отбора.

In [3]:
class ExtraFeatures:

    def __init__(self, items):
        self.titles = items.item_title_raw.fillna('').map(norm).to_numpy()
        self.params = items.item_infm_params_text.fillna('').str[:1200].map(norm).to_numpy()
        self.price = np.log1p(pd.to_numeric(items.item_price, errors='coerce').fillna(0).clip(lower=0).to_numpy('float32'))
        self.rating = items.item_rating.fillna(0).to_numpy('float32')
        self.phone = items.item_is_phone_hidden.to_numpy('float32')
        self.message = items.item_is_message_forbidden.to_numpy('float32')

    def transform(self, row, indices):
        query = norm(row.search_query)
        words = [x for x in query.split() if len(x) > 2]
        params = norm(row.search_infm_params_text)
        pwords = set(params.split())
        threshold = re.search('рейтинг\\D{0,30}([1-5])', params)
        threshold = float(threshold.group(1)) if threshold else 0.0
        result = []
        for j in indices:
            title = self.titles[j]
            tokens = title.split()
            cover = sum((any((t.startswith(w[:5]) for t in tokens)) for w in words)) / max(1, len(words))
            exact = sum((w in tokens for w in words)) / max(1, len(words))
            filter_cover = len(pwords.intersection(self.params[j].split())) / max(1, len(pwords))
            result.append([
                cover, exact, float(bool(query) and query in title),
                filter_cover, float(bool(params)),
                threshold, float(self.rating[j] >= threshold),
                self.price[j], self.phone[j], self.message[j],
            ])
        return np.asarray(result, dtype='float32')

## 2. Сбор кандидатов

Объединяю несколько каналов: TF-IDF по словам и биграммам, символьные 3–5-граммы заголовков, близкие микрокатегории по истории похожих запросов, локальный поиск и точную историю. Сначала беру до 250 / 150 / 200 / 120 объявлений из соответствующих каналов, добавляю совпадения из истории и убираю повторы. Глобальный поиск оставляю, чтобы не потерять дистанционные услуги.

In [4]:
class Retriever:
    """Собирает кандидатов и признаки по тексту, географии и истории."""

    def __init__(self, items, train, extra=False, matrix_format="csc"):
        self.extra = None
        if extra:
            self.extra = ExtraFeatures(items)
        self.items = items.reset_index(drop=True)
        self.n = len(items)
        self.ids = items.item_id.to_numpy()
        self.lookup = {v: k for (k, v) in enumerate(self.ids)}
        self.loc = items.item_location_id.to_numpy()
        self.micro = items.item_microcat_id.to_numpy()
        self.reviews = np.log1p(items.item_rating_reviews_count.fillna(0).to_numpy()).astype('float32')
        self.rating = items.item_rating.fillna(0).to_numpy('float32')
        self.lat = pd.to_numeric(items.item_latitude, errors='coerce').to_numpy('float32')
        self.lon = pd.to_numeric(items.item_longitude, errors='coerce').to_numpy('float32')
        self.centers = items.assign(lat=self.lat, lon=self.lon).groupby('item_location_id')[['lat', 'lon']].median().to_dict('index')
        self.titles = items.item_title_raw.fillna('').map(norm).to_numpy()
        self.exact = defaultdict(Counter)
        self.context = defaultdict(Counter)
        self.pop = np.zeros(self.n, dtype='float32')
        tq = train.search_query.map(norm)
        for (q, l, it) in zip(tq, train.search_location_id, train.item_id):
            j = self.lookup.get(it)
            if j is not None:
                self.exact[q][j] += 1
                self.context[q, l][j] += 1
                self.pop[j] += 1
        self.pop = np.log1p(self.pop)
        log('word TF-IDF')
        self.word = TfidfVectorizer(preprocessor=norm, ngram_range=(1, 2), max_features=180000, min_df=2, sublinear_tf=True, dtype=np.float32)
        self.word.fit(items.text.sample(min(40000, len(items)), random_state=42))
        self.w = self.word.transform(items.text).asformat(matrix_format)
        log('title char TF-IDF')
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=100000, min_df=2, sublinear_tf=True, dtype=np.float32)
        self.c = self.char.fit_transform(self.titles).asformat(matrix_format)
        uq = tq.drop_duplicates().tolist()
        qi = {q: j for (j, q) in enumerate(uq)}
        self.qv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=100000, dtype=np.float32, sublinear_tf=True)
        self.qmat = self.qv.fit_transform(uq).asformat(matrix_format)
        self.qlookup = qi
        cats = sorted(train.item_microcat_id.unique())
        self.cats = np.array(cats)
        ci = {v: j for (j, v) in enumerate(cats)}
        pairs = pd.DataFrame({'q': tq, 'c': train.item_microcat_id}).value_counts().reset_index(name='n')
        self.qcat = sparse.csr_matrix((pairs.n.to_numpy('float32'), ([qi[x] for x in pairs.q], [ci[x] for x in pairs.c])), shape=(len(uq), len(cats)))
        self.qcat = sparse.diags(1 / np.maximum(np.asarray(self.qcat.sum(1)).ravel(), 1)) @ self.qcat
        self.itemci = np.array([ci.get(v, -1) for v in self.micro])
        self.catprior = train.item_microcat_id.value_counts(normalize=True).to_dict()
        log(f'index ready: {self.n} items')

    def features(self, row):
        """Возвращает индексы кандидатов и 17 базовых признаков."""
        q = norm(row.search_query)
        loc = row.search_location_id
        a = (self.w @ self.word.transform([q]).T).toarray().ravel()
        b = (self.c @ self.char.transform([q]).T).toarray().ravel()
        # По похожим запросам оцениваю подходящие микрокатегории
        sim = (self.qmat @ self.qv.transform([q]).T).toarray().ravel()
        ix = top(sim, 30)
        weights = sim[ix] ** 5
        if q in self.qlookup:
            weights[ix == self.qlookup[q]] *= 3
        cp = np.asarray(weights @ self.qcat[ix]).ravel() / max(weights.sum(), 1e-08)
        cat = cp[np.maximum(self.itemci, 0)]
        cat[self.itemci < 0] = 0
        # Близость города это дополнительный сигнал, не жёсткий фильтр
        same = (self.loc == loc).astype('float32')
        center = self.centers.get(loc)
        if center:
            dist = np.sqrt(((self.lat - center['lat']) * 111) ** 2 + ((self.lon - center['lon']) * 111 * np.cos(center['lat'] * np.pi / 180)) ** 2)
            near = np.exp(-np.nan_to_num(dist, nan=10000) / 40)
        else:
            dist = np.full(self.n, 10000)
            near = np.zeros(self.n)
        h = np.zeros(self.n, dtype='float32')
        hc = h.copy()
        for (j, v) in self.exact.get(q, {}).items():
            h[j] = np.log1p(v)
        for (j, v) in self.context.get((q, loc), {}).items():
            hc[j] = np.log1p(v)
        base = (a + 0.6 * b) * (0.15 + 0.85 * np.maximum(same, near * 0.65)) * (0.3 + cat)
        pool = np.unique(np.concatenate([
            top(base, 250),
            top(a + 0.6 * b, 150),
            top((a + 0.6 * b) * (0.1 + same), 200),
            top(cat * (0.1 + same) * (0.05 + a + b), 120),
            np.flatnonzero(h),
        ]))
        j = pool
        f = np.column_stack([
            a[j], b[j], cat[j],
            same[j], near[j],
            np.log1p(np.nan_to_num(dist[j], nan=10000)),
            h[j], hc[j], self.pop[j],
            self.reviews[j], self.rating[j],
            base[j],
            a[j] * same[j], b[j] * same[j], cat[j] * same[j],
            np.full(len(j), len(q.split())),
            np.full(len(j), row.search_is_delivery_search),
        ])
        if self.extra is not None:
            f = np.column_stack([f, self.extra.transform(row, j)])
        return (j, f.astype('float32'))

## 3. Отложенная выборка

Фиксированным seed выбираю 2200 контекстов из `train`: 1386 с новыми текстами и 814 с известными. Для первых удаляю всю историю соответствующих текстов, для вторых - все строки выбранного контекста. На этой очищенной истории строятся все индексы. В корпус локальной проверки добавляю объявления отложенных взаимодействий, которых нет в `benchmark_items`.

In [14]:
train = pd.read_parquet(DATA_DIR / 'train.parquet',
                        columns=SEARCH + ['item_id', 'item_microcat_id'])
benchmark_queries = pd.read_parquet(DATA_DIR / 'benchmark_queries.parquet')

rng = np.random.default_rng(20260928)
normalized = train.search_query.map(norm)
keys = pd.util.hash_pandas_object(train[SEARCH], index=False).to_numpy()
contexts = pd.DataFrame({'q': normalized, 'key': keys}).drop_duplicates()
counts = contexts.q.value_counts()
known_queries = rng.choice(counts[counts >= 2].index.to_numpy(), 814, replace=False)
unseen_queries = rng.choice(np.setdiff1d(contexts.q.unique(), known_queries),
                            1386, replace=False)

chosen_keys = []
for query in np.concatenate([known_queries, unseen_queries]):
    possible_keys = contexts.loc[contexts.q == query, 'key'].to_numpy()
    chosen_keys.append(rng.choice(possible_keys))

held_mask = np.isin(keys, chosen_keys)
held = train.loc[held_mask].copy()
held['key'] = keys[held_mask]
history = train.loc[~held_mask & ~normalized.isin(unseen_queries)].copy()
known_texts = set(history.search_query.map(norm))
known_items = set(history.item_id)

groups = list(held.groupby('key', sort=True))
validation_queries = pd.DataFrame([group.iloc[0] for _, group in groups])
truth = [set(group.item_id) for _, group in groups]
assert len(validation_queries) == 2200
assert len(set(validation_queries.search_query.map(norm))) == 2200

held_items = texts(DATA_DIR / 'train.parquet', set(held.item_id))
validation_items = pd.concat([
    texts(DATA_DIR / 'benchmark_items.parquet'), held_items
], ignore_index=True).drop_duplicates('item_id')

print(f'Контрольные запросы: {len(validation_queries)}')
print(f'История после исключения: {len(history):,} строк')
print(f'Локальный корпус: {len(validation_items):,} объявлений')
del held_items, groups, contexts, counts, keys, normalized, held_mask
gc.collect()

Контрольные запросы: 2200
История после исключения: 489,455 строк
Локальный корпус: 191,404 объявлений


0

## 4. Признаки и локальное сравнение

Для каждого кандидата ставлю метку 1, если объявление есть в отложенных взаимодействиях с запросом. Считаю полноту общего пула и отдельно сравниваю простую формулу, бустинг на 17 основных признаках и бустинг с 10 дополнительными признаками.

In [15]:
retriever = Retriever(validation_items, history, extra=False, matrix_format='csr')
auxiliary = ExtraFeatures(validation_items)
base_features, extended_features, labels, pool_recalls = [], [], [], []

for position, row in enumerate(validation_queries.itertuples(index=False)):
    indices, features = retriever.features(row)
    target = np.array([item in truth[position] for item in retriever.ids[indices]],
                      dtype='uint8')
    base_features.append(features)
    extended_features.append(np.column_stack([
        features, auxiliary.transform(row, indices)
    ]))
    labels.append(target)
    pool_recalls.append(target.sum() / len(truth[position]))
    if (position + 1) % 100 == 0:
        log(f'Кандидаты: {position + 1}/2200; '
            f'средний recall пула: {np.mean(pool_recalls):.4f}')

del retriever, auxiliary, validation_items, history, held, train
gc.collect()

22:06:26 word TF-IDF
22:07:23 title char TF-IDF
22:07:30 index ready: 191404 items
22:07:47 Кандидаты: 100/2200; средний recall пула: 0.9550
22:07:59 Кандидаты: 200/2200; средний recall пула: 0.9275
22:08:11 Кандидаты: 300/2200; средний recall пула: 0.8883
22:08:23 Кандидаты: 400/2200; средний recall пула: 0.8979
22:08:35 Кандидаты: 500/2200; средний recall пула: 0.9053
22:08:48 Кандидаты: 600/2200; средний recall пула: 0.9053
22:09:01 Кандидаты: 700/2200; средний recall пула: 0.9031
22:09:13 Кандидаты: 800/2200; средний recall пула: 0.8973
22:09:25 Кандидаты: 900/2200; средний recall пула: 0.8943
22:09:38 Кандидаты: 1000/2200; средний recall пула: 0.8923
22:09:50 Кандидаты: 1100/2200; средний recall пула: 0.8930
22:10:03 Кандидаты: 1200/2200; средний recall пула: 0.8982
22:10:16 Кандидаты: 1300/2200; средний recall пула: 0.8991
22:10:28 Кандидаты: 1400/2200; средний recall пула: 0.9020
22:10:39 Кандидаты: 1500/2200; средний recall пула: 0.9019
22:10:51 Кандидаты: 1600/2200; средний re

33

Обучение - первые 1400 запросов, сравнение - оставшиеся 800. Для бустинга фиксирую `random_state=42`; основная метрика - средний по запросам Recall@50. Сравниваю также запросы с новыми текстами и с объявлениями, которых не было в истории.

In [16]:
CUT = 1400

def new_model():
    return HistGradientBoostingClassifier(
        max_iter=180, max_leaf_nodes=23, learning_rate=0.07,
        l2_regularization=10, max_bins=128,
        early_stopping=False, random_state=42,
    )

def evaluate(model, feature_list):
    recalls = []
    for i in range(CUT, len(validation_queries)):
        selected = top(model.predict_proba(feature_list[i])[:, 1], 50)
        recalls.append(labels[i][selected].sum() / len(truth[i]))
    return np.asarray(recalls)

training_labels = np.concatenate(labels[:CUT])
base_model = new_model()
base_model.fit(np.concatenate(base_features[:CUT]), training_labels)
base_recalls = evaluate(base_model, base_features)

extended_model = new_model()
extended_model.fit(np.concatenate(extended_features[:CUT]), training_labels)
extended_recalls = evaluate(extended_model, extended_features)

formula_recalls = [
    labels[i][top(base_features[i][:, 11], 50)].sum() / len(truth[i])
    for i in range(CUT, len(validation_queries))
]
seen = validation_queries.search_query.map(norm).isin(known_texts).to_numpy()[CUT:]
cold = np.array([not bool(items & known_items) for items in truth[CUT:]])
metrics = pd.DataFrame([
    ('Формула без обучения', np.mean(formula_recalls)),
    ('Базовый бустинг', base_recalls.mean()),
    ('Расширенный бустинг', extended_recalls.mean()),
    ('Полнота расширенного пула', np.mean(pool_recalls[CUT:])),
    ('Расширенный: известные тексты', extended_recalls[seen].mean()),
    ('Расширенный: новые тексты', extended_recalls[~seen].mean()),
    ('Расширенный: новые объявления', extended_recalls[cold].mean()),
], columns=['Вариант / срез', 'Recall пула'])
print(metrics.to_string(index=False))
print(f'Запросов с полностью новыми объявлениями: {cold.sum()}')

               Вариант / срез  Recall пула
         Формула без обучения     0.731875
              Базовый бустинг     0.813083
          Расширенный бустинг     0.829583
    Полнота расширенного пула     0.903125
Расширенный: известные тексты     0.844807
    Расширенный: новые тексты     0.821565
Расширенный: новые объявления     0.805846
Запросов с полностью новыми объявлениями: 479


## 5. Финальная модель и поиск

Выбранную расширенную модель обучаю на всех 2200 отложенных контекстах. Затем заново строю индексы по полному `train`, но искать объявления разрешаю только в `benchmark_items`. Для локальной оценки выше использовалась модель до этого переобучения.

In [17]:
final_model = new_model()
final_model.fit(np.concatenate(extended_features), np.concatenate(labels))
assert final_model.n_features_in_ == 27
log('Финальная модель обучена на 2200 запросах.')
del base_model, extended_model, base_features, extended_features, labels
gc.collect()

full_history = pd.read_parquet(DATA_DIR / 'train.parquet',
                               columns=SEARCH + ['item_id', 'item_microcat_id'])
benchmark_items = texts(DATA_DIR / 'benchmark_items.parquet')
final_retriever = Retriever(benchmark_items, full_history,
                            extra=True, matrix_format='csc')
del benchmark_items, full_history
gc.collect()

22:12:57 Финальная модель обучена на 2200 запросах.
22:13:09 word TF-IDF
22:14:05 title char TF-IDF
22:14:13 index ready: 189212 items


0

Для каждого запроса оставляю 50 кандидатов с наибольшей оценкой модели. Сохраняю прежний способ сортировки - так не меняется порядок `item_id` в результате.

In [18]:
predictions = []
for position, row in enumerate(benchmark_queries.itertuples(index=False)):
    indices, features = final_retriever.features(row)
    probabilities = final_model.predict_proba(features)[:, 1]
    chosen = indices[top(probabilities, 50)]
    predictions.append(' '.join(final_retriever.ids[chosen]))
    if (position + 1) % 100 == 0:
        log(f'Предсказания: {position + 1}/{len(benchmark_queries)}')

answer = pd.DataFrame({
    'query_id': benchmark_queries.query_id,
    'answer': predictions,
})

22:14:34 Предсказания: 100/2452
22:14:37 Предсказания: 200/2452
22:14:39 Предсказания: 300/2452
22:14:43 Предсказания: 400/2452
22:14:45 Предсказания: 500/2452
22:14:48 Предсказания: 600/2452
22:14:51 Предсказания: 700/2452
22:14:54 Предсказания: 800/2452
22:14:57 Предсказания: 900/2452
22:15:00 Предсказания: 1000/2452
22:15:02 Предсказания: 1100/2452
22:15:05 Предсказания: 1200/2452
22:15:08 Предсказания: 1300/2452
22:15:11 Предсказания: 1400/2452
22:15:14 Предсказания: 1500/2452
22:15:17 Предсказания: 1600/2452
22:15:20 Предсказания: 1700/2452
22:15:23 Предсказания: 1800/2452
22:15:26 Предсказания: 1900/2452
22:15:29 Предсказания: 2000/2452
22:15:32 Предсказания: 2100/2452
22:15:35 Предсказания: 2200/2452
22:15:38 Предсказания: 2300/2452
22:15:41 Предсказания: 2400/2452


## 6. Сохранение ответа

Перед записью проверяю количество строк, уникальность запросов, по 50 разных объявлений на запрос и принадлежность всех ID допустимому корпусу.

In [19]:
assert list(answer.columns) == ['query_id', 'answer']
assert len(answer) == len(benchmark_queries) == 2452
assert answer.query_id.is_unique
assert answer.query_id.str.len().eq(16).all()
assert set(answer.query_id) == set(benchmark_queries.query_id)

valid_ids = set(final_retriever.ids)
for value in answer.answer:
    ids = value.split()
    assert len(ids) == len(set(ids)) == 50
    assert set(ids) <= valid_ids
    assert all(re.fullmatch(r'[0-9a-f]{16}', item) for item in ids)

answer.to_csv(OUTPUT_PATH, index=False, lineterminator='\n')